## K-Means Clustering Example: Clustering WR Archetypes ##

The wide receiver (WR) is one of the most versatile positions in football, both in terms of size and role within a team. Some receivers are used only in the short passing game, while others are used primarily in the redzone or deep balls. In this example, we seek to use K-Means clustering to identify different prototypical groups of WRs, and how these prototypes are used on the field.   

In [1]:
import numpy as np
import rice_ml
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

First, we load in our game data, which contains receiving data from the NFL 2025-2026 season for all WRs who caught any passes. We will also use size data from our players dataset to inform us on how a player's size impacts his usage.  

In [4]:
# Load in NFL receiving stats dataset
csv_path = Path("..") / "data" / "NFLReceivingStats.csv"
receiving = pd.read_csv(csv_path)
# Display the first 5 rows of the dataset
receiving.head(5)

,Rank,Player Id,Player GSIS,Player College GSIS,Player Sportradar Id,Name,Team,#,POS,REC GRD,...,REC 1D,DP,FUM,FUML,CBL,CTT,CTC,YAC,YAC/REC,ADOT
0,NaN,2973,30842,NaN,9c21e9af-681c-41ef-9b00-fbc9e1668ed1,Marcedes Lewis,Broncos,89,TE,55.7,...,0,0,0,0,0,0,0,0,0.0,0.0
1,NaN,7808,39973,NaN,5c48ade7-4b9a-4757-9643-87a6e3839e2b,DeAndre Hopkins,Ravens,10,WR,82.3,...,15,1,0,0,26,19,12,68,2.8,14.5
2,NaN,7816,39983,NaN,de3421f7-2147-4835-89a5-724e87bad463,Zach Ertz,Commanders,86,TE,63.3,...,25,7,1,0,63,16,11,131,2.4,9.4
3,NaN,7844,40011,NaN,c3859e06-5f23-4302-a71b-04820a899d5f,Travis Kelce,Chiefs,87,TE,74.0,...,46,8,1,0,87,9,4,430,5.6,7.1
4,NaN,7857,40024,NaN,5f424505-f29f-433c-b3f2-1a143a04a010,Keenan Allen,Chargers,13,WR,76.1,...,51,8,2,0,96,27,12,252,3.0,8.7


In [7]:
# Load in NFL player stats dataset
csv_path = Path("..") / "data" / "NFL_players.csv"
players = pd.read_csv(csv_path)

# Merge two datasets on pff_id = Player ID
merged = pd.merge(receiving, players, left_on="Player Id", right_on="pff_id")
merged.head(5)

,Rank,Player Id,Player GSIS,Player College GSIS,Player Sportradar Id,Name,Team,#,POS,REC GRD,...,status,ngs_status,ngs_status_short_description,years_of_experience,pff_position,pff_status,draft_year,draft_round,draft_pick,draft_team
0,NaN,2973,30842,NaN,9c21e9af-681c-41ef-9b00-fbc9e1668ed1,Marcedes Lewis,Broncos,89,TE,55.7,...,DEV,DEV,PS; Vet,20,TE,P,2006.0,1.0,28.0,JAX
1,NaN,7808,39973,NaN,5c48ade7-4b9a-4757-9643-87a6e3839e2b,DeAndre Hopkins,Ravens,10,WR,82.3,...,UFA,ACT,Active,14,WR,A,2013.0,1.0,27.0,HOU
2,NaN,7816,39983,NaN,de3421f7-2147-4835-89a5-724e87bad463,Zach Ertz,Commanders,86,TE,63.3,...,UFA,RES,R/Injured,14,TE,A,2013.0,2.0,35.0,PHI
3,NaN,7844,40011,NaN,c3859e06-5f23-4302-a71b-04820a899d5f,Travis Kelce,Chiefs,87,TE,74.0,...,ACT,ACT,Active,14,TE,A,2013.0,3.0,63.0,KC
4,NaN,7857,40024,NaN,5f424505-f29f-433c-b3f2-1a143a04a010,Keenan Allen,Chargers,13,WR,76.1,...,UFA,ACT,Active,14,WR,A,2013.0,3.0,76.0,LAC


Once we have merged the two datasets on the PFF ID as a primary key, we are ready to filter and clean the set. First we filter our data to only WRs. To ensure WR data is not skewed by a small sample of targets, we will remove all receivers with fewer than 15 targets in the season. Then, we select only the relevant ID and feature columns (Name, Player Id, POS, height, weight, YPR, YAC/REC, ADOT, TGT, CTT). In general, these feature columns are per target stats that are independent of volume. This way, WR classification will be impacted only by how a receiver is used when he is on the field. We will be doing some feature engineering with targets (TGT) and Contested Targets (CTT) to see what proportion of a WR's targets come in contested/jump ball situations. We will use that proportion for the model rather than the raw targets and contested targets to avoid volume stats. 

In [ ]:
# Filter for wide receivers only
merged = merged[merged["POS"] == "WR"]
# Filter for players with at least 15 targets
merged = merged[merged["TGT"] >= 15]
# Select relevant features for clustering
features = merged[["Name", "Player Id", "POS", "height", "weight", "YPR", "YAC/REC", "ADOT", "TGT", "CTT"]]
# engineer Coontested target rate (CTR)
features["CTR"] = features["CTT"] / features["TGT"]
# Drop targets and CTT
features = features.drop(columns=["TGT", "CTT"])

features.head(5)


,Name,Player Id,POS,height,weight,YPR,YAC/REC,ADOT,CTR
1,DeAndre Hopkins,7808,WR,73.0,210.0,14.9,2.8,14.5,0.441860
4,Keenan Allen,7857,WR,74.0,211.0,9.8,3.0,8.7,0.207692
6,Adam Thielen,8288,WR,74.0,200.0,9.7,2.5,8.7,0.285714
8,Mike Evans,8642,WR,77.0,231.0,12.8,1.1,14.9,0.376812
9,Brandin Cooks,8655,WR,70.0,190.0,12.3,1.9,17.0,0.361702
...,...,...,...,...,...,...,...,...,...
576,Tetairoa McMillan,158735,WR,77.0,212.0,14.8,3.9,12.6,0.200000
577,Ryan Flournoy,159617,WR,73.0,205.0,12.0,5.2,10.1,0.214286
578,Travis Hunter,160766,WR,73.0,185.0,10.3,4.9,8.1,0.177778
583,Isaiah Bond,163586,WR,71.0,180.0,18.1,5.0,16.4,0.272727


Once we have developed and selected our features, we are ready to standardize our data. We will be using the rice_ml built in Standard Scaler to standardize our features and save the mean and standard deviation of each feature. Once we have finished clustering on the standardized data, we can reconstruct our cluster means for each feature. 

In [11]:
scaler = rice_ml.StandardScaler()
scaled_features = scaler.fit_transform(features[["height", "weight", "YPR", "YAC/REC", "ADOT", "CTR"]])
# print first 5 rows of scaled features
print(scaled_features[:5])

     height    weight       YPR   YAC/REC      ADOT       CTR
1  0.092563  0.724617  0.684879 -0.751213  0.681057  2.512186
4  0.517774  0.789925 -0.891099 -0.627206 -0.914188 -0.120451
6  0.517774  0.071529 -0.922001 -0.937224 -0.914188  0.756711
8  1.793406  2.096101  0.035947 -1.805273  0.791073  1.780873
9 -1.183070 -0.581559 -0.118561 -1.309245  1.368662  1.611005


Now, we are ready to fit our KMeans model. To initialize the KMeans class, we need to provide arguements for the number of clusters (k) and the maximum number of iterations through our dataset. 

We are hypothesizing that we will have three natural types of WRs (short-area/gadget WRs, intermediate route-runners, and deep threats). As a result, we are choosing k = 3. We will use the default max iterations of 100 for now. We also set the seed to 42 outside of the model for result reproducability.

To fit our model, we simply call the fit method with our standardized data as an arguement. This will automatically create three clusters using our KMeans algorithm and save the results. It is important to note that the algorithm uses euclidean distance to fit the model and label points. This is why it is imperative that we use standardized features to avoid feature scale or magnitude issues when labeling points.

In [13]:
# Set seed to 42
np.random.seed(42)

# Initialize KMeans with 3 clusters and a maximum of 100 iterations
kmeans = rice_ml.KMeans(k=3, max_iter=100)

# Fit KMeans to the scaled features
kmeans.fit(scaled_features)

KeyError: "None of [Index([125, 51, 138], dtype='int64')] are in the [columns]"